# Data Visualizations

In [ ]:
# datascience
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')


In [ ]:
# polars
import polars as pl
from IPython.display import display
import hvplot.polars


## Review

In [ ]:
# datascience
united_flights = Table.read_table('united.csv')
united_flights.show(3)


In [ ]:
# polars
united_flights_pl = pl.read_csv('united.csv', eol_char='\r')
united_flights_pl.head(3)


**DISCUSS:** On what fraction of days was flight 73 delayed? 

In [ ]:
# datascience
#  (# of times that 73 was delayed) / (# of times that 73 existed)
flight73 = united_flights.where('Flight Number', 73)
flight73.where('Delay', are.above(0)).num_rows / flight73.num_rows


In [ ]:
# polars
#  (# of times that 73 was delayed) / (# of times that 73 existed)
flight73_pl = united_flights_pl.filter((pl.col('Flight Number') == (73)))
flight73_pl.filter((pl.col('Delay') > (0))).height / flight73_pl.height


**DISCUSS:** Is the Flight Number attribute *numerical* or *categorical*?

## US Census Visualization

In [ ]:
# datascience
full = Table.read_table('nc-est2025-agesex-res.csv')
partial = full.where('AGE', are.below(100)).select('SEX', 'AGE', 'POPESTIMATE2025')
us_pop_2025 = partial.relabeled(2, '2025')
us_pop_2025.show(5)


In [ ]:
# polars
full_pl = pl.read_csv('nc-est2025-agesex-res.csv')
partial_pl = full_pl.filter((pl.col('AGE') < (100))).select('SEX', 'AGE', 'POPESTIMATE2025')
us_pop_2025_pl = partial_pl.rename({partial_pl.columns[2]: '2025'})
us_pop_2025_pl.head(5)


In [ ]:
# datascience
us_pop_2025.where('SEX', 0).plot('AGE', '2025')


In [ ]:
# polars
# hvplot
us_pop_2025_pl.filter((pl.col('SEX') == (0))).hvplot.line(x='AGE', y='2025')


**DISCUSS**: What is one thing interesting about this plot?

In [ ]:
# datascience
us_pop_2025.where('SEX', 0).scatter('AGE', '2025')


In [ ]:
# polars
# hvplot
us_pop_2025_pl.filter((pl.col('SEX') == (0))).hvplot.scatter(x='AGE', y='2025')


**DISCUSS**: What does the expression below compute?

In [ ]:
# datascience
np.average(us_pop_2025.where('SEX', 0).column('AGE'))


In [ ]:
# polars
(us_pop_2025_pl.filter((pl.col('SEX') == (0))).get_column('AGE')).mean()


Compute the average age of people in the US in 2019:

In [ ]:
# datascience
everyone = us_pop_2025.where('SEX', 0)
total_years = sum(everyone.column('AGE') * everyone.column('2025'))
total_people = sum(everyone.column('2025'))
total_years / total_people


In [ ]:
# polars
everyone_pl = us_pop_2025_pl.filter((pl.col('SEX') == (0)))
total_years_pl = sum(everyone_pl.get_column('AGE') * everyone_pl.get_column('2025'))
total_people_pl = sum(everyone_pl.get_column('2025'))
total_years_pl / total_people_pl


### Males vs Females

In [ ]:
# datascience
# Let's compare male and female counts per age
males = us_pop_2025.where('SEX', 1).drop('SEX')
females = us_pop_2025.where('SEX', 2).drop('SEX')


In [ ]:
# polars
# Let's compare male and female counts per age
males_pl = us_pop_2025_pl.filter((pl.col('SEX') == (1))).drop('SEX')
females_pl = us_pop_2025_pl.filter((pl.col('SEX') == (2))).drop('SEX')


In [ ]:
# datascience
pop_2025 = Table().with_columns(
    'Age', males.column('AGE'),
    'Males', males.column('2025'),
    'Females', females.column('2025')
)
pop_2025


In [ ]:
# polars
pop_2025_pl = pl.DataFrame([
    (males_pl.get_column('AGE')).alias('Age'),
    (males_pl.get_column('2025')).alias('Males'),
    (females_pl.get_column('2025')).alias('Females')
])
pop_2025_pl


In [ ]:
# datascience
pop_2025.plot('Age')


In [ ]:
# polars
# hvplot
(lambda _df: _df.hvplot.line(x='Age', y=[c for c in _df.columns if c != 'Age']))(pop_2025_pl)


**Discuss**: In a sentence, what data are represented by this line plot?

## Scatter Plots ##

Data from 2017: 50 top actors with the highest total gross domestic box office

In [ ]:
# datascience
# Actors and their highest grossing movies
actors = Table.read_table('actors.csv')
actors


In [ ]:
# polars
# Actors and their highest grossing movies
actors_pl = pl.read_csv('actors.csv', eol_char='\r').with_columns(pl.col('Total Gross').str.strip_chars().cast(pl.Float64), pl.col('Average per Movie').str.strip_chars().cast(pl.Float64), pl.col('Gross').str.strip_chars().cast(pl.Float64))
actors_pl


In [ ]:
# datascience
actors.scatter('Number of Movies', 'Total Gross')


In [ ]:
# polars
# hvplot
actors_pl.hvplot.scatter(x='Number of Movies', y='Total Gross')


In [ ]:
# datascience
actors.scatter('Number of Movies', 'Average per Movie')


In [ ]:
# polars
# hvplot
actors_pl.hvplot.scatter(x='Number of Movies', y='Average per Movie')


Who is the outlier?

In [ ]:
# datascience
actors.where('Average per Movie', are.above(400))


In [ ]:
# polars
actors_pl.filter((pl.col('Average per Movie') > (400)))


## Bar Charts ##

In [ ]:
# datascience
# Highest grossing movies as of 2017
top_movies = Table.read_table('top_movies_2017.csv')
top_movies


In [ ]:
# polars
# Highest grossing movies as of 2017
top_movies_pl = pl.read_csv('top_movies_2017.csv', eol_char='\r')
top_movies_pl


In [ ]:
# datascience
top10_adjusted = top_movies.take(np.arange(10))
top10_adjusted


In [ ]:
# polars
top10_adjusted_pl = top_movies_pl.gather(pl.int_range(10, eager=True))
top10_adjusted_pl


In [ ]:
# datascience
# Convert to millions of dollars for readability
millions = np.round(top10_adjusted.column('Gross (Adjusted)') / 1000000, 3)
top10_adjusted = top10_adjusted.with_column('Millions', millions)
top10_adjusted


In [ ]:
# polars
# Convert to millions of dollars for readability
millions_pl = (top10_adjusted_pl.get_column('Gross (Adjusted)') / 1000000).round(3)
top10_adjusted_pl = top10_adjusted_pl.with_columns(pl.Series('Millions', millions_pl))
top10_adjusted_pl


In [ ]:
# datascience
# A line plot doesn't make sense here: don't do this!
top10_adjusted.plot('Year', 'Millions')


In [ ]:
# polars
# hvplot
# A line plot doesn't make sense here: don't do this!
top10_adjusted_pl.hvplot.line(x='Year', y='Millions')


In [ ]:
# datascience
top10_adjusted.barh('Title', 'Millions')


In [ ]:
# polars
# matplotlib
plots.barh(top10_adjusted_pl.get_column('Title'), top10_adjusted_pl.get_column('Millions'))


**Exercise**: Generate the chart shown in the slides: a bar chart of age (# years since release) for the 10 highest grossing movies (non-adjusted).

In [ ]:
# datascience
# Get a table of the top 10 grossing movies (not adjusted for increasing ticket prices)
#  - sort then take *
# Compute the age of each of these movies
#  - get an array of the years they were released and subtract from 2026 *
# Put it all together in a bar chart
#  - use with_column to put ages into the table and then use barh

top10 = top_movies.sort('Gross', descending=True).take(np.arange(10))
ages = 2026 - top10.column('Year')
top10_with_age = top10.with_column('Age in 2026', ages)
top10_with_age.barh('Title', 'Age in 2026')


In [ ]:
# polars
# matplotlib
# Get a table of the top 10 grossing movies (not adjusted for increasing ticket prices)
#  - sort then take *
# Compute the age of each of these movies
#  - get an array of the years they were released and subtract from 2026 *
# Put it all together in a bar chart
#  - use with_column to put ages into the table and then use barh

top10_pl = top_movies_pl.sort('Gross', descending=True).gather(pl.int_range(10, eager=True))
ages_pl = 2026 - top10_pl.get_column('Year')
top10_with_age_pl = top10_pl.with_columns(pl.Series('Age in 2026', ages_pl))
plots.barh(top10_with_age_pl.get_column('Title'), top10_with_age_pl.get_column('Age in 2026'))
